In [1]:
import pathlib
import ollama
from itertools import product
import re
import tiktoken

In [2]:
OUTPUT = '../data/output.txt'
OUTPUT_MD = '../data/output.md'
PROMPTS = '../prompts/prompts.txt'

In [ ]:
system_prompt = """
Role:
Experienced racing driver coach.

Task:
Analyze driving data and give short, precise feedback.
You receive a table with attributes.
So that you can perform your analysis, in addition to the player’s driving data you receive an optimally driven reference lap for comparison.
Each attribute in the table has two values: first value = driver, second value = reference.

Notes:
Take into account the units of the respective attributes. They are specified in the table.
Keep in mind that the user neither knows the optimal driving line nor values that refer to the x, y, and z axes.
They only know throttle, brake, steering, speed, position on the track, and timestamp.
The user cannot do much with concrete timestamps, but rather thinks in sections of the track.

Examples:
'You drove this segment without errors; however, in the preceding segment you were too slow and could no longer reach a sufficient top speed.',
'Your line was too wide on the outside, which led to a longer path and less grip. Try turning in earlier to find a better racing line.',
'You have to take the corner at 110 instead of 160 km/h in order not to crash.',
'You drove this section almost perfectly, keep it up!'

Negative examples:
'The strongly negative acceleration_x (-34 vs. reference 0) together with a high Z value (-77 vs. -20) indicates excessive braking and understeer, causing the speed to drop to 96 km/h instead of the optimal 228 km/h.',
'From about 9 s onward you reach lateral accelerations of up to +20 m/s², while the reference lap only allows 0–5 m/s² – this creates strong under-/oversteer. Later (from 21 s onward) you brake too late, causing the speed to drop abruptly from 198 → 96 km/h and the vehicle to deviate from the ideal line at yaw values around –0.5 rad.' 
'In the first minutes, eexcessive negative acceleration_x and incorrectly dosed yaw rotation dominate, which leads to repeated slowing; from about 68 s onward the acceleration is chosen positively and the speed can increase again. The goal is to reduce braking so that the vehicle can take the corner with a similar speed as in the reference value (e.g. 173 km/h at 70.9 s).'
"""

user_prompt = ""

In [4]:
# Count tokens for System prompt and User prompt
def count_tokens(prompt: str, model_name: str = "gpt-4") -> int:
    """
    Zählt die Tokens für einen Prompt für das angegebene Modell.
    """
    encoding = tiktoken.encoding_for_model(model_name)
    tokens = encoding.encode(prompt)
    return len(tokens)

text = []

with open('../data/output.md', "r", encoding="utf-8") as f:
    for line in f:
        line = line.rstrip("\n")
        if line != '\'':
            text.append(line + '\n')

sys_tokens = count_tokens(system_prompt, model_name="gpt-4")  # ersetze ggf. durch dein Modell
print(f"Token-Anzahl System: {sys_tokens}")
usr_tokens = count_tokens(user_prompt, model_name="gpt-4")  # ersetze ggf. durch dein Modell
print(f"Token-Anzahl User {usr_tokens}")

Token-Anzahl System: 485
Token-Anzahl User 0


In [ ]:
# Helper functions
def log_response(timestamps, lap, segment, system_prompt, user_prompt, resp):
    # -- Logging ---
    if(segment == 0):
        log_round(system_prompt, user_prompt, resp)
    log_segment_response(timestamps, lap, segment, resp)

def log_round(system_prompt, user_prompt, resp):
    # --- Logging ---
    pathlib.Path("prompts").mkdir(parents=True, exist_ok=True)
    with open(PROMPTS, "a", encoding="utf-8") as file:
        file.writelines([
            "XXXX" * 80 + "\n",
            "System Prompt: " + system_prompt + "\n",
            "User Prompt: " + user_prompt + "\n",
        ])

def log_segment_response(timestamps, lap, segment, resp):
    min_timestamp = min(timestamps)
    max_timestamp = max(timestamps)

    pathlib.Path("prompts").mkdir(parents=True, exist_ok=True)
    with open(PROMPTS, "a", encoding="utf-8") as file:
        file.writelines([
            "__" * 80 + "\n",
            f"Lap: {lap}, Segment: {segment}, Sequence: {min_timestamp} - {max_timestamp}\n",
            "Response: " + resp["message"]["content"] + "\n\n",
        ])

def get_segment_md(md_row: str) -> int:
    return int(md_row.split("|")[-2].strip())

def get_lap_md(md_row: str) -> int:
    return int(md_row.split("|")[-3].strip())

def get_timestamp_md(md_row: str) -> float:
    ts_cell = md_row.split("|")[2].strip()
    return float(ts_cell.strip("()").split(",")[0])

# TXT Approach

In [ ]:
text = []

with open(OUTPUT, "r", encoding="utf-8") as f:
    for line in f:
        line = line.rstrip("\n")
        if line != '\'':
            text.append(line + '\n')

# list(product([0], [0, 1, 2, 3, 4, 5])) + list(product([1,2], [0,1,2,3,4]))
for lap, segment in list(product([0], [0, 1, 2, 3, 4])):
    print(f'Lap: {lap}, Segment: {segment}')
    
    # Filter text for lap and segment
    filterd_lines = [row for row in text if f"'lap_number': {lap}" in row and f"'segment': {segment}" in row]
    filtered_text = '\n'.join(filterd_lines)   
    
    # # --- RAG setup --- TODO
    # retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    # question = "How do I setup my car best in Forza Horizon 5?"
    # retrieved_docs = retriever.invoke(question)
    # retrieved_docs[2].page_content
    #rag_context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    
    # Generate LLM text from output file
    resp = ollama.chat(
        model="nemotron-3-nano:30b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt + f"\n```json\n{filtered_text}\n```"},
        ],
    )

    # Loggging
    data_tokens = count_tokens(filtered_text, model_name="gpt-4")
    print(f"Token-Anzahl User {data_tokens}")
    timestamps = []
    for line in filterd_lines:
        num = r"[-+]?(?:\d*\.\d+|\d+\.?\d*)(?:[eE][-+]?\d+)?"
        pattern = rf"timestamp in s'\s*:\s*({num})\s*,\s*({num})"
        timestamps += [float(re.search(pattern, line).group(1))]
    log_response(timestamps, lap, segment, system_prompt, user_prompt, resp)

# MD Table Approach

In [ ]:
def get_segment_information(segment_rows)-> str:
    num = r"[-+]?(?:\d*\.\d+|\d+\.?\d*)(?:[eE][-+]?\d+)?"
    pair_re = re.compile(rf"\(\s*({num})\s*,\s*({num})\s*\)")

    # lists for user / optimal (reference)
    timestamps_user = []
    timestamps_opt = []
    speeds_user = []
    speeds_opt = []
    yaws_user = []
    yaws_opt = []

    for row in segment_rows:
        cells = [c.strip() for c in row.split("|")]

        # timestamp usually in column index 2
        if len(cells) > 2:
            m = pair_re.search(cells[2])
            if m:
                timestamps_user.append(float(m.group(1)))
                timestamps_opt.append(float(m.group(2)))

        # yaw usually in column index 5
        if len(cells) > 5:
            m = pair_re.search(cells[5])
            if m:
                yaws_user.append(float(m.group(1)))
                yaws_opt.append(float(m.group(2)))

        # speed usually in column index 9
        if len(cells) > 10:
            m = pair_re.search(cells[10])
            if m:
                speeds_user.append(float(m.group(1)))
                speeds_opt.append(float(m.group(2)))

    # --- Timestamp / duration ---
    if timestamps_user:
        start_user = min(timestamps_user)
        end_user = max(timestamps_user)
        duration_user = end_user - start_user
    else:
        start_user = end_user = duration_user = 0.0

    if timestamps_opt:
        start_opt = min(timestamps_opt)
        end_opt = max(timestamps_opt)
        duration_opt = end_opt - start_opt
    else:
        start_opt = end_opt = duration_opt = 0.0

    # --- Speed stats ---
    max_speed_user = max(speeds_user) if speeds_user else 0.0
    min_speed_user = min(speeds_user) if speeds_user else 0.0
    max_speed_opt = max(speeds_opt) if speeds_opt else 0.0
    min_speed_opt = min(speeds_opt) if speeds_opt else 0.0

    # --- Yaw stats (use absolute values for extremes) ---
    max_yaw_user = max((abs(v) for v in yaws_user), default=0.0)
    max_yaw_opt = max((abs(v) for v in yaws_opt), default=0.0)

    # --- Build summary ---
    ret = (
        f"Time — user: {duration_user:.3f}s, "
        f"ref: {duration_opt:.3f}s (Δ {duration_user - duration_opt:+.3f}s). "
    )

    if speeds_user or speeds_opt:
        ret += (
            f"Max speed — user: {max_speed_user:.1f} km/h, ref: {max_speed_opt:.1f} km/h; "
            f"Min speed — user: {min_speed_user:.1f} km/h, ref: {min_speed_opt:.1f} km/h. "
        )
    else:
        ret += "No speed data. "

    if yaws_user or yaws_opt:
        ret += f"Max yaw (abs) — user: {max_yaw_user:.3f}°, ref: {max_yaw_opt:.3f}°."
    else:
        ret += "No yaw data."

    print(ret)
    return ret

In [ ]:
text = []

# --- Read markdown file ---
with open(OUTPUT_MD, "r", encoding="utf-8") as f:
    lines = [line.rstrip("\n") for line in f if line.strip()]

# --- Split header and rows ---
header = lines[0]
separator = lines[1]
rows = lines[2:]

# --- Process per lap / segment ---
for lap, segment in product([0], [0, 1, 2, 3, 4]):
    print("Lap:" + str(lap) + ", Segment:" + str(segment))
    segment_rows = [r for r in rows if get_segment_md(r) == segment and get_lap_md(r) == lap]

    if not segment_rows:
        continue
    
    md_block = "\n".join([header, separator, *segment_rows])
    print(md_block)

    segment_info = get_segment_information(segment_rows)

    # --- LLM call ---
    resp = ollama.chat(
        model="glm-4.7-flash:q4_K_M",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt + f"\n```markdown\n{md_block}\n```" + f"\n\nSegment Summary:\n{segment_info}"},
        ],
    )

    # Logging
    data_tokens = count_tokens(md_block, model_name="gpt-4")
    print(f"Token-Anzahl User {data_tokens}")
    timestamps = [get_timestamp_md(r) for r in segment_rows]
    log_response(timestamps, lap, segment, system_prompt, user_prompt, resp)

Lap:0, Segment:0
|      | timestamp in s   | acceleration_x in m/s²   | acceleration_y in m/s²   | acceleration_z in m/s²   | yaw in degrees   | position_x in m   | position_y in m   | position_z in m   | speed in km/h   | lap_number   |   segment |
|-----:|:-----------------|:-------------------------|:-------------------------|:-------------------------|:-----------------|:------------------|:------------------|:------------------|:----------------|:-------------|----------:|
|    0 | (0.0, 0.0)       | (0, 0)                   | (0, 0)                   | (0, 0)                   | (-2.5, -2.5)     | (981, 981)        | (303, 303)        | (2664, 2664)      | (0, 0)          | (0, 0)       |         0 |
|   19 | (0.983, 0.052)   | (0, 0)                   | (0, 0)                   | (2, 0)                   | (-2.5, -2.5)     | (981, 981)        | (303, 303)        | (2664, 2664)      | (1, 0)          | (0, 0)       |         0 |
|   38 | (1.99, 0.828)    | (0, 0)                 